In [1]:
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import gc

@dataclass
class SequenceStore:
    # Per-row arrays
    species_ids: np.ndarray
    gene_ids: np.ndarray
    sequences: list[np.ndarray]
    # Lookup tables
    species_to_gene_to_rows: dict[int, dict[int, list[int]]]
    valid_species: np.ndarray
    # Name lookups
    species_names: list[str]
    gene_names: list[str]
    # Token map used for pre-encoding
    token_to_id: dict[str, int]

    @classmethod
    def from_parquet(cls, parquet_path: str | Path, order_col: str = "order", family_col: str = "family",
                     genus_col: str = "genus", species_col: str = "species", gene_col: str = "gene", 
                     sequence_col: str = "sequence", select_orders: list[str] | None = None,
                     select_families: list[str] | None = None, select_genera: list[str] | None = None,
                     select_species: list[str] | None = None, select_genes: list[str] | None = None) -> "SequenceStore":
        print("[SequenceStore] - Loading Parquet file")
        df = pd.read_parquet(parquet_path, columns=[order_col, family_col, genus_col, species_col, gene_col,
                                                    sequence_col])

        # Filter by taxonomy or genes if provided
        for select_list, select_col in [
            (select_orders, order_col), (select_families, family_col),(select_genera, genus_col),
            (select_species, species_col), (select_genes, gene_col)]:
            if select_list is not None:
                print(f"[SequenceStore] - Filtering column {select_col} for {select_list}")
                df = df[df[select_col].isin(select_list)]
                
                # Optional: Check if any rows remain
                if df.empty:
                    raise ValueError("No rows remain after filtering")

        # Basic checks
        print("[SequenceStore] - Checking for missing values")
        if df[species_col].isna().any():
            raise ValueError(f"Column '{species_col}' contains missing values.")
        if df[gene_col].isna().any():
            raise ValueError(f"Column '{gene_col}' contains missing values.")
        if df[sequence_col].isna().any():
            raise ValueError(f"Column '{sequence_col}' contains missing values.")
        
        # Ensure categorical dtype, and remove unused categories to ensure continuous numbering
        print("[SequenceStore] - Ensuring continuous categorical species and gene columns")
        species_cat = df[species_col].astype('category').cat.remove_unused_categories()
        gene_cat = df[gene_col].astype('category').cat.remove_unused_categories()

        # Extract category integer IDs
        print("[SequenceStore] - Extracting category integer IDs")
        species_ids = species_cat.cat.codes.to_numpy(copy=True)
        gene_ids = gene_cat.cat.codes.to_numpy(copy=True)

        # Extract category names
        print("[SequenceStore] - Extracting category names")
        species_names = species_cat.cat.categories.to_list()
        gene_names = gene_cat.cat.categories.to_list()

        #Pre-encode sequences, using tiny vocabulary for DNA / ambiguous bases
        print("[SequenceStore] - Pre-encoding sequences, using tiny vocabulary for DNA / ambiguous bases ")
        token_to_id = {
            'PAD': 0,
            'A': 1,
            'C': 2,
            'G': 3,
            'T': 4,
            'N': 5,
        }

        encode_array = np.full(256, token_to_id['N'], dtype=np.uint8)
        for token, idx in token_to_id.items():
            if token != 'PAD':
                encode_array[ord(token)] = idx

        def encode_dna(seq):
        # Convert string to ASCII bytes, then use the encode_array as a lookup table
            return encode_array[np.frombuffer(seq.encode('ascii'), dtype=np.uint8)]
        
        sequences = [encode_dna(seq) for seq in df[sequence_col].array]
        
        # Build nested row-index mapping
        print("[SequenceStore] - Building nested row-index mapping")
        species_to_gene_to_rows = defaultdict(lambda: defaultdict(list))
        for row_idx, (sid, gid) in enumerate(zip(species_ids, gene_ids)):
            species_to_gene_to_rows[int(sid)][int(gid)].append(row_idx)

        species_to_gene_to_rows = {
            sid: dict(gene_map)
            for sid, gene_map in species_to_gene_to_rows.items()
        }

        valid_species = np.array(sorted(species_to_gene_to_rows.keys()))

        print("[SequenceStore] - Cleaning up temporary memory")
        del df, species_cat, gene_cat
        gc.collect()

        return cls(
            species_ids=species_ids,
            gene_ids=gene_ids,
            sequences=sequences,
            species_to_gene_to_rows=species_to_gene_to_rows,
            valid_species=valid_species,
            species_names=species_names,
            gene_names=gene_names,
            token_to_id=token_to_id
        )
    
    @property
    def num_rows(self) -> int:
        return len(self.sequences)
    
    @property
    def num_species(self) -> int:
        return len(self.species_names)
    
    @property
    def num_valid_species(self) -> int:
        return len(self.valid_species)
    
    @property
    def num_genes(self) -> int:
        return len(self.gene_names)
    
    @property
    def vocab_size(self) -> int:
        return len(self.token_to_id)
    
    def summary(self) -> str:
        return (
            f"SequenceStore(num_rows={self.num_rows}, "
            f"num_species={self.num_species}, "
            f"num_genes={self.num_genes}, "
            f"valid_species={self.num_valid_species})"
        )

In [2]:
select_orders = ['Cypriniformes', 'Perciformes', 'Siluriformes', 'Gobiiformes']
select_families = None
select_genera = None
select_species = None
select_genes = None

In [3]:
path_processed_dataset = Path("output") / "processed_dataset.parquet"
store = SequenceStore.from_parquet(path_processed_dataset, select_orders=select_orders, select_families=select_families,
                                   select_genera=select_genera, select_species=select_species,
                                   select_genes=select_genes)
store.summary()

[SequenceStore] - Loading Parquet file
[SequenceStore] - Filtering column order for ['Cypriniformes', 'Perciformes', 'Siluriformes', 'Gobiiformes']
[SequenceStore] - Checking for missing values
[SequenceStore] - Ensuring continuous categorical species and gene columns
[SequenceStore] - Extracting category integer IDs
[SequenceStore] - Extracting category names
[SequenceStore] - Pre-encoding sequences, using tiny vocabulary for DNA / ambiguous bases 
[SequenceStore] - Building nested row-index mapping
[SequenceStore] - Cleaning up temporary memory


'SequenceStore(num_rows=193241, num_species=4571, num_genes=15, valid_species=4571)'

In [ ]:
from dataclasses import dataclass
import numpy as np
import torch


@dataclass
class BatchBuilder:
    store: SequenceStore
    species_per_batch: int = 64
    crop_length: int = 512
    rng_seed: int | None = None
    pin_memory: bool = False

    def __post_init__(self) -> None:
        self.rng = np.random.default_rng(self.rng_seed)
        self.pad_id = self.store.token_to_id["PAD"]

        if self.species_per_batch < 1:
            raise ValueError("species_per_batch must be at least 1.")
        if self.crop_length < 1:
            raise ValueError("crop_length must be at least 1.")
        if self.species_per_batch > len(self.store.valid_species):
            raise ValueError(
                "species_per_batch cannot exceed the number of valid species."
            )

    @property
    def batch_size(self) -> int:
        return 2 * self.species_per_batch

    def sample_species(self) -> np.ndarray:
        return self.rng.choice(
            self.store.valid_species,
            size=self.species_per_batch,
            replace=False,
        )

    def sample_two_rows_for_species(self, species_id: int) -> tuple[tuple[int, int], tuple[int, int]]:
        gene_to_rows = self.store.species_to_gene_to_rows[species_id]
        gene_ids = list(gene_to_rows.keys())

        # Case A: species has >= 2 genes
        if len(gene_ids) >= 2:
            gene1, gene2 = self.rng.choice(gene_ids, size=2, replace=False)
            row1 = int(self.rng.choice(gene_to_rows[gene1]))
            row2 = int(self.rng.choice(gene_to_rows[gene2]))
            return (row1, gene1), (row2, gene2)

        # Only one gene available
        gene = gene_ids[0]
        rows = gene_to_rows[gene]

        # Case B: one gene, >= 2 rows
        if len(rows) >= 2:
            row1, row2 = self.rng.choice(rows, size=2, replace=False)
            return (int(row1), gene), (int(row2), gene)

        # Case C: one gene, one row
        row = rows[0]
        return (row, gene), (row, gene)

    def write_crop_into_numpy_arrays(self, seq: np.ndarray, input_ids: np.ndarray, attention_mask: np.ndarray,
                                     batch_idx: int) -> None:
        """
        Write one cropped/padded sequence into preallocated NumPy arrays.

        Parameters
        ----------
        seq
            1D uint8 encoded sequence
        input_ids
            Preallocated NumPy array of shape [B, L], dtype=int64
        attention_mask
            Preallocated NumPy array of shape [B, L], dtype=bool
        batch_idx
            Which row in the batch to fill
        """
        L = self.crop_length
        seq_len = len(seq)

        if seq_len >= L:
            if seq_len == L:
                crop = seq
            else:
                start = self.rng.integers(0, seq_len - L + 1)
                crop = seq[start:start + L]

            input_ids[batch_idx, :] = crop
            attention_mask[batch_idx, :] = True
            return

        # Shorter than crop length: pad
        input_ids[batch_idx, :] = self.pad_id
        attention_mask[batch_idx, :] = False

        input_ids[batch_idx, :seq_len] = seq
        attention_mask[batch_idx, :seq_len] = True

    def sample_batch_cpu(self) -> dict[str, torch.Tensor]:
        """
        Build one batch on CPU using NumPy first, then convert once to torch.
        """
        sampled_species = self.sample_species()
        B = self.batch_size
        L = self.crop_length

        input_ids_np = np.full((B, L), fill_value=self.pad_id, dtype=np.int64)
        attention_mask_np = np.zeros((B, L), dtype=bool)
        species_ids_np = np.empty(B, dtype=np.int64)
        gene_ids_np = np.empty(B, dtype=np.int64)

        batch_idx = 0
        for species_id in sampled_species:
            species_id = int(species_id)
            view1, view2 = self.sample_two_rows_for_species(species_id)

            for row_idx, gene_id in (view1, view2):
                seq = self.store.sequences[row_idx]
                self.write_crop_into_numpy_arrays(
                    seq=seq,
                    input_ids=input_ids_np,
                    attention_mask=attention_mask_np,
                    batch_idx=batch_idx,
                )
                species_ids_np[batch_idx] = species_id
                gene_ids_np[batch_idx] = int(gene_id)
                batch_idx += 1

        input_ids = torch.from_numpy(input_ids_np)
        attention_mask = torch.from_numpy(attention_mask_np)
        species_ids = torch.from_numpy(species_ids_np)
        gene_ids = torch.from_numpy(gene_ids_np)

        if self.pin_memory and torch.cuda.is_available():
            input_ids = input_ids.pin_memory()
            attention_mask = attention_mask.pin_memory()
            species_ids = species_ids.pin_memory()
            gene_ids = gene_ids.pin_memory()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "species_ids": species_ids,
            "gene_ids": gene_ids,
        }

    def move_batch_to_device(self, batch: dict[str, torch.Tensor], device: str | torch.device,
                             non_blocking: bool = True) -> dict[str, torch.Tensor]:
        device = torch.device(device)
        return {
            k: v.to(device, non_blocking=non_blocking)
            for k, v in batch.items()
        }

In [5]:
builder = BatchBuilder(
    store=store,
    species_per_batch=64,
    crop_length=512,
    rng_seed=None,
    pin_memory=True
)

batch = builder.sample_batch_cpu()

for i in range(0, len(batch["species_ids"]), 2):
    sid1 = int(batch["species_ids"][i].item())
    sid2 = int(batch["species_ids"][i + 1].item())
    gid1 = int(batch["gene_ids"][i].item())
    gid2 = int(batch["gene_ids"][i + 1].item())

    print(
        f"pair {i//2}: species={store.species_names[sid1]}, "
        f"genes=({store.gene_names[gid1]}, {store.gene_names[gid2]})"
    )

pair 0: species=Luciobarbus nasus, genes=(Cytb, ATPase6)
pair 1: species=Pareiorhina carrancas, genes=(12S_rRNA, COXI)
pair 2: species=Etheostoma zonistium, genes=(Cytb, COXI)
pair 3: species=Channallabes apus, genes=(Cytb, COXI)
pair 4: species=Lycodes palearis, genes=(Cytb, ND6)
pair 5: species=Pterygoplichthys etentaculatus, genes=(12S_rRNA, 16S_rRNA)
pair 6: species=Gobius xanthocephalus, genes=(12S_rRNA, Cytb)
pair 7: species=Trypauchenopsis limicola, genes=(16S_rRNA, ND2)
pair 8: species=Osteochilus nashii, genes=(12S_rRNA, ND3)
pair 9: species=Helicolenus dactylopterus, genes=(ND5, ND6)
pair 10: species=Epinephelus undulosus, genes=(ND2, ND3)
pair 11: species=Garra kempi, genes=(ND5, ATPase6)
pair 12: species=Pronotogrammus martinicensis, genes=(ND1, COXIII)
pair 13: species=Schizolecis guntheri, genes=(ND2, 16S_rRNA)
pair 14: species=Thoburnia hamiltoni, genes=(ND4, ND5)
pair 15: species=Hisonotus charrua, genes=(Cytb, 16S_rRNA)
pair 16: species=Hoplisoma similis, genes=(Cytb, 

In [ ]:
from torch.utils.data import IterableDataset, DataLoader, get_worker_info


class ContrastiveBatchDataset(IterableDataset):
    """
    IterableDataset that keeps your existing BatchBuilder logic and lets
    DataLoader workers prebuild full batches in the background.
    """

    def __init__(self, store: SequenceStore, species_per_batch: int = 64, crop_length: int = 512,
                 base_seed: int | None = None, pin_memory: bool = False) -> None:
        super().__init__()
        self.store = store
        self.species_per_batch = species_per_batch
        self.crop_length = crop_length
        self.base_seed = base_seed
        self.pin_memory = pin_memory

    def __iter__(self):
        worker = get_worker_info()

        if worker is None:
            worker_id = 0
        else:
            worker_id = worker.id

        rng_seed = self.base_seed if self.base_seed is None else self.base_seed + worker_id
        # Each worker gets its own RNG seed, so they do not build identical batches
        builder = BatchBuilder(
            store=self.store,
            species_per_batch=self.species_per_batch,
            crop_length=self.crop_length,
            rng_seed=rng_seed,
            pin_memory=self.pin_memory,
        )

        while True:
            yield builder.sample_batch_cpu()


def make_batch_loader(store: SequenceStore, species_per_batch: int = 64, crop_length: int = 512,
                      num_workers: int = 4, pin_memory: bool = True, prefetch_factor: int = 2,
                      persistent_workers: bool = True, base_seed: int = 12345) -> DataLoader:
    dataset = ContrastiveBatchDataset(
        store=store,
        species_per_batch=species_per_batch,
        crop_length=crop_length,
        base_seed=base_seed,
        pin_memory=pin_memory,
    )

    loader_kwargs = dict(
        dataset=dataset,
        batch_size=None,   # dataset already yields complete batches
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    if num_workers > 0:
        loader_kwargs["prefetch_factor"] = prefetch_factor
        loader_kwargs["persistent_workers"] = persistent_workers

    return DataLoader(**loader_kwargs)

In [6]:
import torch.nn as nn
import torch.nn.functional as F


class SequenceEncoder(nn.Module):
    """
    Transformer-based encoder for DNA sequences with optional gene embeddings.

    Input:
        input_ids:      [B, L] torch.long
        attention_mask: [B, L] torch.bool
            True  = real token
            False = padding
        gene_ids:       [B] torch.long
            gene identity for each sequence

    Output:
        z_seq: [B, embed_dim]
            L2-normalized sequence embeddings
    """

    def __init__(self, vocab_size: int, max_length: int, num_genes: int, d_model: int = 256, nhead: int = 8,
                 num_layers: int = 4, dim_feedforward: int = 1024, dropout: float = 0.1, embed_dim: int = 256,
                 pad_token_id: int = 0, use_gene_embedding: bool = True) -> None:
        super().__init__()

        if d_model % nhead != 0:
            raise ValueError("d_model must be divisible by nhead.")

        self.vocab_size = vocab_size
        self.max_length = max_length
        self.num_genes = num_genes
        self.d_model = d_model
        self.embed_dim = embed_dim
        self.pad_token_id = pad_token_id
        self.use_gene_embedding = use_gene_embedding

        # Token embedding
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=d_model,
            padding_idx=pad_token_id,
        )

        # Positional embedding
        self.position_embedding = nn.Embedding(
            num_embeddings=max_length,
            embedding_dim=d_model,
        )

        # Optional gene embedding
        if self.use_gene_embedding:
            self.gene_embedding = nn.Embedding(
                num_embeddings=num_genes,
                embedding_dim=d_model,
            )

        # Input dropout
        self.input_dropout = nn.Dropout(dropout)

        # Transformer encoder stack
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_layers,
        )

        # Projection head
        self.projection = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, embed_dim),
        )

    def masked_mean_pool(self, x: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """
        x: [B, L, d_model]
        attention_mask: [B, L] bool, True=real token, False=padding
        """
        mask = attention_mask.unsqueeze(-1).to(dtype=x.dtype)  # [B, L, 1]
        x_masked = x * mask
        summed = x_masked.sum(dim=1)  # [B, d_model]
        counts = mask.sum(dim=1).clamp(min=1.0)  # [B, 1]
        return summed / counts

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                gene_ids: torch.Tensor | None = None) -> torch.Tensor:
        """
        input_ids:      [B, L] torch.long
        attention_mask: [B, L] torch.bool
        gene_ids:       [B] torch.long or None

        returns:
            z_seq: [B, embed_dim], L2-normalized
        """
        if input_ids.ndim != 2:
            raise ValueError(f"input_ids must have shape [B, L], got {input_ids.shape}")
        if attention_mask.ndim != 2:
            raise ValueError(
                f"attention_mask must have shape [B, L], got {attention_mask.shape}"
            )
        if input_ids.shape != attention_mask.shape:
            raise ValueError(
                f"input_ids and attention_mask must have same shape, got "
                f"{input_ids.shape} and {attention_mask.shape}"
            )

        batch_size, seq_len = input_ids.shape

        if seq_len > self.max_length:
            raise ValueError(
                f"Sequence length {seq_len} exceeds max_length={self.max_length}"
            )

        if self.use_gene_embedding:
            if gene_ids is None:
                raise ValueError("gene_ids must be provided when use_gene_embedding=True")
            if gene_ids.ndim != 1 or gene_ids.shape[0] != batch_size:
                raise ValueError(
                    f"gene_ids must have shape [B], got {gene_ids.shape}"
                )

        # Position indices: [B, L]
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(
            batch_size, seq_len
        )

        # Token + position embeddings
        x = self.token_embedding(input_ids) + self.position_embedding(positions)

        # Add gene embedding if enabled
        if self.use_gene_embedding:
            gene_vec = self.gene_embedding(gene_ids)          # [B, d_model]
            gene_vec = gene_vec.unsqueeze(1)                  # [B, 1, d_model]
            x = x + gene_vec                                  # broadcast over L

        # Input dropout
        x = self.input_dropout(x)

        # PyTorch transformer expects True for positions to ignore
        src_key_padding_mask = ~attention_mask  # [B, L]

        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)  # [B, L, d_model]

        pooled = self.masked_mean_pool(x, attention_mask)    # [B, d_model]
        z_seq = self.projection(pooled)                      # [B, embed_dim]
        z_seq = F.normalize(z_seq, dim=-1)

        return z_seq

In [7]:
class SpeciesContrastiveModel(nn.Module):
    """
    Sequence encoder + species prototype table.

    Training objective:
        - batch-restricted prototype loss
        - half-vs-half CLIP-style sequence contrastive loss
    """

    def __init__(self, encoder: SequenceEncoder, num_species: int, embed_dim: int, temperature: float = 0.07,
                 lambda_proto: float = 1.0, lambda_seq: float = 1.0) -> None:
        super().__init__()

        if temperature <= 0:
            raise ValueError("temperature must be > 0")

        self.encoder = encoder
        self.temperature = temperature
        self.lambda_proto = lambda_proto
        self.lambda_seq = lambda_seq

        self.species_embedding = nn.Embedding(num_species, embed_dim)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, species_ids: torch.Tensor,
                gene_ids: torch.Tensor) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
        """
        Loss = batch-restricted prototype loss + half-vs-half sequence contrastive loss

        Assumes the batch is organized in paired views:
            0,1   = same species
            2,3   = same species
            4,5   = same species
            ...

        Parameters
        ----------
        input_ids : [B, L]
        attention_mask : [B, L]
        species_ids : [B]
        gene_ids : [B]

        Returns
        -------
        total_loss : scalar tensor
        outputs : dict of tensors for logging/debugging
        """
        # ------------------------------------------------------------
        # 1) Encode sequences
        # ------------------------------------------------------------
        z_seq = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            gene_ids=gene_ids,
        )  # [B, D], already normalized by encoder

        B = z_seq.shape[0]
        device = z_seq.device

        if B % 2 != 0:
            raise ValueError(f"Batch size must be even, got {B}")

        # ------------------------------------------------------------
        # 2) Batch-restricted prototype loss
        # ------------------------------------------------------------
        batch_species_ids, local_targets = torch.unique(
            species_ids,
            sorted=True,
            return_inverse=True,
        )  # batch_species_ids: [S], local_targets: [B]

        z_species_batch = self.species_embedding(batch_species_ids)  # [S, D]
        z_species_batch = F.normalize(z_species_batch, dim=-1)

        logits_proto = (z_seq @ z_species_batch.T) / self.temperature  # [B, S]
        loss_proto = F.cross_entropy(logits_proto, local_targets)

        # ------------------------------------------------------------
        # 3) Half-vs-half CLIP-style sequence contrastive loss
        # ------------------------------------------------------------
        # Split into the two views
        z_a = z_seq[0::2]   # [S, D]
        z_b = z_seq[1::2]   # [S, D]

        S = z_a.shape[0]
        targets_seq = torch.arange(S, device=device)  # diagonal targets

        logits_seq = (z_a @ z_b.T) / self.temperature  # [S, S]

        # Symmetric CLIP-style loss
        loss_seq_ab = F.cross_entropy(logits_seq, targets_seq)
        loss_seq_ba = F.cross_entropy(logits_seq.T, targets_seq)
        loss_seq = 0.5 * (loss_seq_ab + loss_seq_ba)

        # ------------------------------------------------------------
        # 4) Combine losses
        # ------------------------------------------------------------
        total_loss = self.lambda_proto * loss_proto + self.lambda_seq * loss_seq

        outputs = {
            "z_seq": z_seq,
            "batch_species_ids": batch_species_ids,   # [S]
            "z_species_batch": z_species_batch,       # [S, D]
            "logits_proto": logits_proto,             # [B, S]
            "logits_seq": logits_seq,                 # [S, S]
            "loss_proto": loss_proto.detach(),
            "loss_seq": loss_seq.detach(),
            "local_targets": local_targets,           # [B]
            "targets_seq": targets_seq,               # [S]
        }

        return total_loss, outputs

In [8]:
import time
from pathlib import Path

import torch


def topk_accuracy(logits: torch.Tensor, targets: torch.Tensor, k: int = 1) -> torch.Tensor:
    """
    Compute top-k accuracy.

    logits:  [N, C]
    targets: [N]
    """
    k = min(k, logits.shape[1])
    topk = logits.topk(k, dim=1).indices
    correct = (topk == targets.unsqueeze(1)).any(dim=1)
    return correct.float().mean()


def save_checkpoint(save_path: str | Path, model: SpeciesContrastiveModel, optimizer: torch.optim.Optimizer,
                    step: int, loss: float, loss_proto: float, loss_seq: float, acc_proto_top1: float,
                    acc_proto_top5: float, acc_seq: float, species_names: list[str]) -> None:
    """
    Save a training checkpoint including model weights and the current
    species embedding matrix for downstream clustering analysis.
    """
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    checkpoint = {
        "step": step,
        "loss": loss,
        "loss_proto": loss_proto,
        "loss_seq": loss_seq,
        "acc_proto_top1": acc_proto_top1,
        "acc_proto_top5": acc_proto_top5,
        "acc_seq": acc_seq,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "species_embeddings": model.species_embedding.weight.detach().cpu(),
        "species_names": species_names,
    }

    torch.save(checkpoint, save_path)


def train_species_contrastive(model: SpeciesContrastiveModel, builder: BatchBuilder, optimizer: torch.optim.Optimizer,
                              device: str | torch.device, num_steps: int, checkpoint_dir: str | Path,
                              checkpoint_every: int = 500, log_every: int = 50, grad_clip_norm: float | None = 1.0,
                              start_step: int = 0, use_amp: bool = True) -> list[dict]:
    """
    Train the species-contrastive model with optional AMP.

    Periodically saves checkpoints containing:
    - encoder + species embedding weights
    - optimizer state
    - current species embedding matrix
    - metadata (step, losses, accuracies)

    Returns
    -------
    history : list of dict
        Per-log-step training metrics.
    """
    device = torch.device(device)
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    amp_enabled = use_amp and device.type == "cuda"
    scaler = torch.GradScaler("cuda", enabled=amp_enabled)

    model.train()
    history = []

    t0 = time.time()

    for step in range(start_step + 1, start_step + num_steps + 1):
        # Build batch on CPU, then move once to GPU
        batch_cpu = builder.sample_batch_cpu()
        batch = builder.move_batch_to_device(batch_cpu, device=device, non_blocking=True)

        with torch.autocast("cuda", enabled=amp_enabled):
            loss, outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                species_ids=batch["species_ids"],
                gene_ids=batch["gene_ids"],
            )

        logits_proto = outputs["logits_proto"]   # [B, S]
        logits_seq = outputs["logits_seq"]       # [S, S]
        local_targets = outputs["local_targets"] # [B]
        targets_seq = outputs["targets_seq"]     # [S]

        # Metrics are computed outside autocast for consistency/readability
        acc_proto_top1 = topk_accuracy(logits_proto.float(), local_targets, k=1)
        acc_proto_top5 = topk_accuracy(logits_proto.float(), local_targets, k=5)
        acc_seq = topk_accuracy(logits_seq.float(), targets_seq, k=1)

        loss_proto = outputs["loss_proto"]
        loss_seq = outputs["loss_seq"]

        optimizer.zero_grad(set_to_none=True)

        if amp_enabled:
            scaler.scale(loss).backward()

            if grad_clip_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()

            if grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

            optimizer.step()

        # Logging
        if step % log_every == 0 or step == start_step + 1:
            elapsed = time.time() - t0
            metrics = {
                "step": step,
                "loss": float(loss.item()),
                "loss_proto": float(loss_proto.item()),
                "loss_seq": float(loss_seq.item()),
                "acc_proto_top1": float(acc_proto_top1.item()),
                "acc_proto_top5": float(acc_proto_top5.item()),
                "acc_seq": float(acc_seq.item()),
                "elapsed_sec": elapsed,
            }
            history.append(metrics)

            print(
                f"[step {step:6d}] "
                f"loss={metrics['loss']:.4f} "
                f"proto={metrics['loss_proto']:.4f} "
                f"seq={metrics['loss_seq']:.4f} "
                f"acc1={metrics['acc_proto_top1']:.4f} "
                f"acc5={metrics['acc_proto_top5']:.4f} "
                f"acc_seq={metrics['acc_seq']:.4f} "
                f"time={elapsed:.1f}s"
            )

        # Periodic checkpointing
        if step % checkpoint_every == 0 or step == start_step + num_steps:
            save_path = checkpoint_dir / f"step_{step:07d}.pt"
            save_checkpoint(
                save_path=save_path,
                model=model,
                optimizer=optimizer,
                step=step,
                loss=float(loss.item()),
                loss_proto=float(loss_proto.item()),
                loss_seq=float(loss_seq.item()),
                acc_proto_top1=float(acc_proto_top1.item()),
                acc_proto_top5=float(acc_proto_top5.item()),
                acc_seq=float(acc_seq.item()),
                species_names=builder.store.species_names,
            )
            print(f"Saved checkpoint to {save_path}")

    return history

In [9]:
checkpoint_dir: str = "output/checkpoints_species_contrastive_top4orders"
resume_ckpt_path: str | None = "output/checkpoints_species_contrastive_top4orders/step_0838000.pt"
num_steps: int = 162_000

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = SequenceEncoder(
    vocab_size=store.vocab_size,
    max_length=builder.crop_length,
    num_genes=store.num_genes,
    d_model=256,
    nhead=8,
    num_layers=4,
    dim_feedforward=1024,
    dropout=0.1,
    embed_dim=256,
    pad_token_id=store.token_to_id["PAD"],
    use_gene_embedding=True
).to(device)

model = SpeciesContrastiveModel(
    encoder=encoder,
    num_species=store.num_valid_species,
    embed_dim=256,
    temperature=0.07,
    lambda_proto=1.0,
    lambda_seq=1.0
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-2,
)

start_step = 0
if resume_ckpt_path is not None:
    checkpoint = torch.load(resume_ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_step = checkpoint["step"]

print(f"Resuming from step {start_step}")

history = train_species_contrastive(
    model=model,
    builder=builder,
    optimizer=optimizer,
    device=device,
    num_steps=num_steps,
    checkpoint_dir=checkpoint_dir,
    checkpoint_every=1000,
    log_every=50,
    grad_clip_norm=1.0,
    start_step=start_step,
    use_amp=True
)

/tmp/ipykernel_525520/3483181054.py:71: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Resuming from step 838000
[step 838001] loss=0.8195 proto=0.2336 seq=0.5859 acc1=0.8672 acc5=1.0000 acc_seq=0.7812 time=1.0s
[step 838050] loss=1.0881 proto=0.2992 seq=0.7889 acc1=0.8828 acc5=1.0000 acc_seq=0.7031 time=25.2s
[step 838100] loss=0.8898 proto=0.3171 seq=0.5727 acc1=0.8984 acc5=1.0000 acc_seq=0.7812 time=50.0s
[step 838150] loss=0.4360 proto=0.0648 seq=0.3712 acc1=1.0000 acc5=1.0000 acc_seq=0.8906 time=68.7s
[step 838200] loss=0.8875 proto=0.2074 seq=0.6801 acc1=0.9375 acc5=1.0000 acc_seq=0.7812 time=80.5s
[step 838250] loss=0.6923 proto=0.1601 seq=0.5322 acc1=0.9297 acc5=1.0000 acc_seq=0.8281 time=92.3s
[step 838300] loss=0.7358 proto=0.1871 seq=0.5486 acc1=0.9688 acc5=1.0000 acc_seq=0.8438 time=104.1s
[step 838350] loss=0.6882 proto=0.1709 seq=0.5173 acc1=0.9375 acc5=1.0000 acc_seq=0.7969 time=115.9s
[step 838400] loss=0.7677 proto=0.2101 seq=0.5576 acc1=0.9453 acc5=1.0000 acc_seq=0.7969 time=127.7s
[step 838450] loss=0.7107 proto=0.2016 seq=0.5091 acc1=0.9062 acc5=1.000